# Compare logical state-preparation circuits

Fault-tolerant quantum computations consume code blocks prepared in logical states such as $|0_L\rangle$ and $|+_L\rangle$, so errors left by a noisy preparation circuit affect every later step that uses its output.
The [fault-tolerance certificates notebook](../logical_error_rates/5_fault_tolerance_certificates.ipynb#Bounded-fault-propagation-in-state-preparation) checks whether a few faults can leave a heavy error on the prepared block, but this yes-or-no answer cannot rank two circuits that both pass.
The [state-preparation notebook](../logical_error_rates/4_state_preparation.ipynb#Flagged-state-preparation) instead estimates a logical error rate: how often a decoder mispredicts a logical operator that should stabilize the prepared state.

A logical error rate misses two kinds of error that matter to the circuits that use the block next.
First, it checks only the logical operators that stabilize the prepared state; for the $|0_L\rangle$ state of a code with one logical qubit, that is $Z_L$, which no Z-type error can flip.
Z-type errors therefore affect this benchmark only by confusing the decoder about other errors, even though later circuits must still correct them.
Second, an error that the decoder corrects does not count as a failure, even though it uses up part of the code's capacity to correct further errors later.

Below, we estimate a figure of merit that accounts for both kinds of error.
It needs no decoder, counts errors of both Pauli types, and weighs each error by how likely it is to make the next circuit fail.
These tools come from `qldpc.experimental`, so their interface may still change.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim

from qldpc import circuits, codes, decoders
from qldpc.experimental.circuits import (
    ConsumerProfile,
    estimate_downstream_fault_weight,
    estimate_residual_weight_histogram,
)

## A preparation circuit to benchmark

We benchmark the flagged preparation of the Steane code's $|0_L\rangle$ state from Figure 1c of [Goto (2016)](https://www.nature.com/articles/srep19578), under circuit-level depolarizing noise at physical error rate $p = 10^{-3}$.
Its flag qubit, qubit 7, detects single circuit errors that would otherwise leave a heavy error on the block.
A repeat-until-success protocol prepares the block again whenever the flag fires.
The options `add_flags=True` and `post_select="flags"` model this protocol by adding a detector for the flag measurement and discarding the shots in which it fires.
By default, the benchmark takes the prepared block to be the qubits that the circuit acts on but never measures, here qubits 0 through 6.

A code of distance $d$ corrects every error of weight at most $t = \lfloor (d - 1) / 2 \rfloor$, which is $t = 1$ for the Steane code.

In [3]:
code = codes.SteaneCode()
prep_circuit = stim.Circuit("""
    # non-fault-tolerant preparation of the logical zero state
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag qubit 7 detects errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")
error_rate = 1e-3
noise_model = circuits.DepolarizingNoiseModel(error_rate)
noisy_circuit = noise_model.noisy_circuit(prep_circuit)
options = {"add_flags": True, "post_select": "flags"}
correctable_weight = (code.get_distance() - 1) // 2

## Logical error rate and bounded propagation

We first apply the two existing benchmarks to this circuit.
Stim describes the circuit's noise as a list of error mechanisms, independent random events such as a Pauli error after a gate, and a fault is the set of error mechanisms that occur in one shot.
The bounded-propagation certificate checks every fault of at most $t$ error mechanisms that passes post-selection.
The logical error rate uses the same lookup decoder and flag post-selection as the state-preparation notebook.

In [4]:
certificate = circuits.certify_bounded_fault_propagation(
    noisy_circuit, max_weight=correctable_weight, **options
)
print(f"bounded fault propagation up to fault weight {correctable_weight}: {certificate.certified}")

num_samples = 4 * 10**7  # fewer samples give a quicker, noisier estimate


def get_logical_error_rate(circuit: stim.Circuit) -> float:
    # sample the logical error rate of a noisy preparation circuit, post-selecting on its flags;
    # the code's qubits are taken to be qubits 0, 1, ..., len(code) - 1 of the circuit
    diagnostic_circuit, detector_record = circuits.get_state_prep_diagnostic_circuit(
        code, circuit, add_flags=True
    )
    logical_error_rate, _ = circuits.get_logical_error_and_discard_rate(
        diagnostic_circuit,
        decoders.lookup(max_weight=correctable_weight),
        num_samples=num_samples,
        post_select=detector_record.get_events("flags"),
    )
    return logical_error_rate


print(f"logical error rate: {get_logical_error_rate(noisy_circuit):.1e}")

bounded fault propagation up to fault weight 1: True
logical error rate: 1.4e-05


The certificate is exact within the circuit's noise model: every accepted fault of at most $t$ error mechanisms leaves an error no heavier than its number of error mechanisms, up to stabilizers of the prepared state.
The logical error rate is a sampled estimate, with a statistical uncertainty of about 5% here.
Neither result says how often the block carries a Z-type error, or a correctable error, into the next circuit.

## Residual weights

A fault leaves a Pauli error on the prepared block.
Multiplying that error by a stabilizer of the prepared state does not change the state, so what matters is the smallest weight among all such equivalent errors, which we call the residual weight of the fault.
For a CSS state such as this one, weight means by default the CSS weight $\max(|x|, |z|)$ of a Pauli error with X support $x$ and Z support $z$; `css_weight=False` instead counts the qubits that the error acts on.
A residual weight of zero means that the shot prepared the ideal state, while a residual weight above $t$ exceeds what the code is guaranteed to correct.

`estimate_residual_weight_histogram` estimates the distribution of the residual weight over the shots that pass post-selection.
It computes the contributions of faults with few error mechanisms exactly, samples the rarer faults with more error mechanisms, and reports first-order standard errors for the sampled part.

In [5]:
histogram = estimate_residual_weight_histogram(
    noisy_circuit, max_residual_weight=correctable_weight, seed=0, **options
)
print(f"probability that post-selection keeps a shot: {histogram.acceptance_probability:.4f}")
for weight, probability in enumerate(histogram.get_residual_weight_distribution()):
    label = f"{weight}" if weight <= correctable_weight else f"> {correctable_weight}"
    print(f"  residual weight {label}: {probability:.2e}")

probability that post-selection keeps a shot: 0.9925
  residual weight 0: 9.94e-01
  residual weight 1: 6.33e-03
  residual weight > 1: 1.05e-05


Post-selection keeps about 99.25% of shots, so about 0.75% of preparations are repeated.
Most kept shots prepare the ideal state.
About six in a thousand leave an error of residual weight one, which the lookup decoder usually corrects, so these errors rarely show up in the logical error rate.
A residual weight above $t$ is hundreds of times rarer, because the certificate rules it out for every accepted fault of a single error mechanism.

## A single figure of merit

How much a residual error matters depends on the circuit that uses the block next, which we call its consumer, such as a round of fault-tolerant error correction.
A fault-tolerant consumer is designed to correct every combination of errors of total weight at most $t$, and its own faults count against that budget.
We model the consumer pessimistically: a block of residual weight $m$ survives a consumer in which $g$ error mechanisms occur exactly when $m + g \le t$.
If the faults of the preparation and of the consumer are independent, the probability that the block's residual error causes a failure that a perfect block would have avoided is

$$
\Phi = \Pr[m + g > t] - \Pr[g > t].
$$

A real decoder also corrects many heavier errors, so $\Phi$ measures how often the residual error breaks the consumer's guarantee, rather than how often a real consumer fails.

When the consumer has many independent error mechanisms, each with a small probability, $g$ is approximately Poisson-distributed with mean $\nu$, the expected number of error mechanisms that occur in the consumer.
`estimate_downstream_fault_weight` estimates $\nu$ for a reference consumer, by default one round of Steane-type error correction.
That round couples the block to fresh code blocks through $2n$ transversal two-qubit gates and measures $2n$ qubits, so $\nu \approx 2n(p_2 + p_m)$, where $n$ is the number of qubits in the block and $p_2$ and $p_m$ are the error probabilities of a two-qubit gate and a measurement.

In [6]:
expected_fault_weight = estimate_downstream_fault_weight(noisy_circuit, code=code)
consumer = ConsumerProfile.poisson(expected_fault_weight, correctable_weight)
figure_of_merit, standard_error = histogram.get_excess_failure_probability(consumer)
print(f"expected fault weight of the consumer: nu = {expected_fault_weight:.3f}")
print(
    f"figure of merit: Phi = {figure_of_merit:.2e} (sampling standard error {standard_error:.0e})"
)

fault_free_consumer = ConsumerProfile.hard(correctable_weight)
residual_tail, _ = histogram.get_excess_failure_probability(fault_free_consumer)
print(f"with a fault-free consumer (nu = 0): Phi = Pr[m > t] = {residual_tail:.2e}")

expected fault weight of the consumer: nu = 0.028
figure of merit: Phi = 1.83e-04 (sampling standard error 5e-14)
with a fault-free consumer (nu = 0): Phi = Pr[m > t] = 1.05e-05


Here $\nu = 0.028$ is the expected number of error mechanisms that occur in one round of the reference consumer, not a failure probability.
With this consumer, $\Phi$ is more than ten times larger than with a fault-free consumer, because a residual error of weight one together with a single consumer error already exceeds $t = 1$.
$\Phi$ is therefore dominated by how often the block carries any residual error at all, which the logical error rate does not measure.
The sampling standard error is negligible because the faults that dominate $\Phi$ are computed exactly; it does not account for the uncertainty of the consumer model or of neglected rare faults.

## Phase noise after preparation

To see how the two benchmarks respond to Z-type errors, the next cell adds a Z error with probability $p$ to each qubit of the block after the preparation, as an idle period might, and recomputes the logical error rate and $\Phi$.

In [7]:
z_idling = stim.Circuit(f"Z_ERROR({error_rate}) " + " ".join(map(str, range(len(code)))))
idle_circuit = noisy_circuit + z_idling
idle_histogram = estimate_residual_weight_histogram(
    idle_circuit, max_residual_weight=correctable_weight, seed=0, **options
)
idle_figure_of_merit, _ = idle_histogram.get_excess_failure_probability(consumer)
for label, block_histogram in [("without", histogram), ("with", idle_histogram)]:
    residual_error = 1 - block_histogram.get_residual_weight_distribution()[0]
    print(f"{label} Z idling: probability of a residual error {residual_error:.2e}")
print(f"with Z idling: logical error rate {get_logical_error_rate(idle_circuit):.1e}")
print(f"with Z idling: Phi = {idle_figure_of_merit:.2e}")

without Z idling: probability of a residual error 6.34e-03
with Z idling: probability of a residual error 1.33e-02
with Z idling: logical error rate 1.8e-05
with Z idling: Phi = 3.71e-04


The idle noise roughly doubles the probability that the block carries a residual error, and $\Phi$ roughly doubles with it.
The logical error rate also rises, but only through decoder confusion: the lookup table explains each syndrome with at most one error mechanism, so an idle Z error in the same shot as a correctable error that flips $Z_L$ can make the decoder mispredict $Z_L$.
A decoder that inferred X-type errors from the Z-type stabilizers alone would not be affected, whereas $\Phi$ counts Z-type errors directly through the residual weight, whatever the decoder.
For the Steane code, every Z-type error on $|0_L\rangle$ is equivalent to one of weight at most one, so this noise matters only in combination with other errors, which $\Phi$ captures through the consumer.

## Comparing preparation protocols

$\Phi$ is defined over the shots that pass post-selection, so we report it together with the acceptance probability, which sets how often a preparation must be repeated.
The next cell compares the flagged protocol with the same circuit run without post-selection on its flag.

In [8]:
for post_select in ["flags", False]:
    protocol_options = {**options, "post_select": post_select}
    certified = circuits.certify_bounded_fault_propagation(
        noisy_circuit, max_weight=correctable_weight, **protocol_options
    ).certified
    protocol_histogram = estimate_residual_weight_histogram(
        noisy_circuit, max_residual_weight=correctable_weight, seed=0, **protocol_options
    )
    protocol_figure_of_merit, _ = protocol_histogram.get_excess_failure_probability(consumer)
    print(
        f"post_select={post_select!r}: certified {certified}, "
        f"keeps {protocol_histogram.acceptance_probability:.4f} of shots, "
        f"Phi {protocol_figure_of_merit:.2e}"
    )

post_select='flags': certified True, keeps 0.9925 of shots, Phi 1.83e-04
post_select=False: certified False, keeps 1.0000 of shots, Phi 2.14e-03


Without post-selection, the circuit is not certified and $\Phi$ is more than ten times larger, in exchange for never repeating a preparation.
Pairs of $\Phi$ and acceptance probability make it possible to compare candidate circuits and post-selection rules on both axes.

## Adapting this notebook

You can benchmark another preparation by changing the code, circuit, noise model, and acceptance rule.
The circuit must be a noisy Stim Clifford circuit that starts in Stim's all-zero state and leaves its output qubits in a stabilizer state.
`post_select` accepts the same options as `circuits.certify_bounded_fault_propagation`, including detectors already present in the circuit.
The logical-error-rate helper and the phase-noise cell assume that the code's qubits are the first `len(code)` qubits of the circuit, and the helper post-selects on added flags; adjust them for other layouts or acceptance rules.
For a code whose distance is not already known, computing $t$ can require an expensive exact distance search.
`max_residual_weight` sets the largest residual weight that the histogram resolves; use at least $t$, and more to score consumers that tolerate heavier residual errors.

The default $\nu$ reads error probabilities from the noisy circuit and counts the operations of one round of Steane-type error correction.
For another consumer, pass its expected fault weight to `ConsumerProfile.poisson`, or its distribution of fault weights to `ConsumerProfile.from_downstream_fault_weights`, which can also include the residual weights of other prepared blocks that it uses.
A consumer profile can also be built from measured failure probabilities of a real consumer and decoder, replacing the pessimistic model above.

$\Phi$ rests on several modeling assumptions: independent faults in the preparation and the consumer, a consumer that fails exactly when the total weight exceeds $t$, a Poisson-distributed number of consumer errors, and a value of $\nu$ that counts only the gates and measurements of the reference consumer.
It is a figure of merit for comparing circuits, not a predicted logical error rate.
The standard errors of the residual-weight histogram cover only its sampled part, and they can be optimistic when the outcomes that matter are rare among the samples.
The [API reference](https://qldpc.readthedocs.io/autoapi/qldpc/experimental/circuits/benchmarking/index.html) defines these quantities precisely and states the remaining limitations.